In [1]:
from glob import glob
import pandas as pd
import os
import soundfile as sf
from tqdm import tqdm
from multiprocess import Pool
from scipy.io import wavfile
import itertools
import io
import numpy as np
import json
import re
import zipfile
from pathlib import Path

def chunks(l, n):
    for i in range(0, len(l), n):
        yield (l[i: i + n], i // n)

def multiprocessing(strings, function, cores=6, returned=True):
    df_split = chunks(strings, len(strings) // cores)
    pool = Pool(cores)
    pooled = pool.map(function, df_split)
    pool.close()
    pool.join()

    if returned:
        return list(itertools.chain(*pooled))

/usr/lib/python3/dist-packages/scipy/__init__.py:146: UserWarning: A NumPy version >=1.17.3 and <1.25.0 is required for this version of SciPy (detected version 1.26.4
  warnings.warn(f"A NumPy version >={np_minversion} and <{np_maxversion}"


In [3]:
# from huggingface_hub import snapshot_download

# snapshot_download(
#     repo_id="MCAA1-MSU/anv_data_ke", 
#     repo_type="dataset", 
#     local_dir="./anv_data_ke",
#     allow_patterns="*/train/*/audios/*.parquet"
# )

In [4]:
files = glob('anv_data_ke/*/train/*/audios/*.parquet')
len(files)

1038

In [5]:
def loop(files):

    os.environ['OMP_NUM_THREADS'] = '1'
    os.environ['OPENBLAS_NUM_THREADS'] = '1'
    
    files, _ = files

    data = []
    for f in tqdm(files):
        base = '_'.join(f.split('/')[:2]) + '_audio'
        f_new = f.replace('/', '-').replace('.parquet', '')
        os.makedirs(base, exist_ok=True)
        df = pd.read_parquet(f)
        for i in range(len(df)):
            try:
                t = df['transcription'].iloc[i].strip()
                if len(t) < 2:
                    continue
                audio_filename = f'{f_new}_{i}.mp3'
                audio_filename = os.path.join(base, audio_filename)
                b = df['audio'].iloc[i]['bytes']
                audio_np, sr = sf.read(io.BytesIO(b))
                if audio_np.ndim > 1:
                    audio_np = audio_np.mean(axis=1)
                if audio_np.shape[0] < 10000:
                    continue
                sf.write(audio_filename, audio_np, sr)
                
                data.append({
                    'audio_filename': audio_filename,
                    'text': t,
                    'speaker': f"{base}"
                })
            except Exception as e:
                pass
        
    return data

In [6]:
# data = loop((files[:1], 0))

In [7]:
data = multiprocessing(files, loop, cores = 20)

100%|██████████| 51/51 [1:46:11<00:00, 124.93s/it]


In [8]:
from datasets import Dataset

dataset = Dataset.from_list(data)
dataset[0]

{'audio_filename': 'anv_data_ke_luo_audio/anv_data_ke-luo-train-unscripted-audios-train_unscripted_007_0.mp3',
 'text': "Olemoni otow, bathe konchiel otow. To mae [?] inyalo kel kod kute mawuotho, kata koso goyone yath e sama owinjore. To mae inyalo geng' kod yath ma igoyo dinwoya dinwoya mar mondo mi ogeng' kute manyalo molo yadhno, manyalo molo olemono ma kelne chandruok kabila no.",
 'speaker': 'anv_data_ke_luo_audio'}

In [10]:
dataset.push_to_hub('malaysia-ai/Multilingual-TTS', 'anv_data_ke')

Creating parquet from Arrow format: 100%|██████████| 2/2 [00:00<00:00,  7.23ba/s]
Processing Files (0 / 0): |          |  0.00B /  0.00B            
Processing Files (0 / 1):  33%|███▎      | 16.8MB / 51.5MB, 1.91MB/s  
Processing Files (0 / 1): 100%|█████████▉| 51.4MB / 51.5MB, 5.71MB/s  
Processing Files (1 / 1): 100%|██████████| 51.5MB / 51.5MB, 5.36MB/s  
Processing Files (1 / 1): 100%|██████████| 51.5MB / 51.5MB, 5.25MB/s  
New Data Upload: 100%|██████████| 51.5MB / 51.5MB, 5.25MB/s  
Uploading the dataset shards: 100%|██████████| 1/1 [00:10<00:00, 10.91s/ shards]


CommitInfo(commit_url='https://huggingface.co/datasets/malaysia-ai/Multilingual-TTS/commit/b4e820b3c32c27a5ee6123d31bb83cb815446478', commit_message='Upload dataset', commit_description='', oid='b4e820b3c32c27a5ee6123d31bb83cb815446478', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/malaysia-ai/Multilingual-TTS', endpoint='https://huggingface.co', repo_type='dataset', repo_id='malaysia-ai/Multilingual-TTS'), pr_revision=None, pr_num=None)

In [12]:
audio_files = [d['audio_filename'] for d in data]

a = list(set(audio_files))
with open('anv_data_ke-audio.json', 'w') as fopen:
    json.dump(a, fopen)

pd.DataFrame({'audio': a}).to_parquet('anv_data_ke-audio.parquet')

In [7]:
# for f in glob('anv_data_ke_*_audio'):
#     print(f)
#     os.system(f'zip -rq {f}.zip {f}')

In [6]:
# for f in glob('anv_data_ke_*_audio.zip'):
#     print(f)
#     os.system(f'hf upload malaysia-ai/Multilingual-TTS {f} --repo-type=dataset')

In [12]:
# for f in glob('anv_data_ke_*_audio_neucodec'):
#     print(f)
#     os.system(f'zip -rq {f}.zip {f}')

In [13]:
# for f in glob('anv_data_ke_*_audio_neucodec.zip'):
#     print(f)
#     os.system(f'hf upload malaysia-ai/Multilingual-TTS {f} --repo-type=dataset')